# Cleaning The Dataset
- identifying the important features and converting them to usable data structure (string to one hot encoding)

In [15]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import numpy as np
from sklearn.model_selection import KFold,train_test_split
from xgboost import XGBClassifier

In [2]:
data = pd.read_csv("dataset/job_placement.csv")

In [3]:
data.head()

,id,name,gender,age,degree,stream,college_name,placement_status,salary,gpa,years_of_experience
0,1,John Doe,Male,25,Bachelor's,Computer Science,Harvard University,Placed,60000,3.7,2.0
1,2,Jane Smith,Female,24,Bachelor's,Electrical Engineering,Massachusetts Institute of Technology,Placed,65000,3.6,1.0
2,3,Michael Johnson,Male,26,Bachelor's,Mechanical Engineering,Stanford University,Placed,58000,3.8,3.0
3,4,Emily Davis,Female,23,Bachelor's,Information Technology,Yale University,Not Placed,0,3.5,2.0
4,5,David Brown,Male,24,Bachelor's,Computer Science,Princeton University,Placed,62000,3.9,2.0


In [4]:
data['college_name'].value_counts().head()

college_name
University of California--Berkeley          43
University of Michigan--Ann Arbor           43
University of Virginia                      43
University of Illinois--Urbana-Champaign    43
University of Colorado--Boulder             43
Name: count, dtype: int64

In [5]:
# conversion of target column to numeric binary classification
def convert_target(x):
    if x.lower() == "placed":
        return 1
    else:
        return 0

data['target'] = data['placement_status'].apply(convert_target)

In [6]:
# Multiplying the GPA to 2 because in colleges we have gpa out of 10
data['gpa'] = data['gpa']*2

# Analysis of the dataset
- the dataset contains a column with stream which we can convert to numerical by label encoding (as its unique)
- the dataset also contains a column with salary after placement which we can convert to simple binary column by keeping the value as one where salary>mean_of_salary
- the dataset also contains various college name for which i am gonna convert that column to frequency reputation , that is frequency of people placed through a college over totatl number of students to get the reputation.

In [7]:
stream_encoder = LabelEncoder()
stream_encoder.fit(data['stream'])
data['stream_encoded']=stream_encoder.transform(data['stream'])

In [8]:
mean_of_salary = data['salary'].mean()
def check(x):
    if x>=mean_of_salary.round(-2):
        return 1
    else:
        return 0
data[f'salary>{mean_of_salary.round(-2)}'] = data['salary'].apply(check)

In [9]:
# target enc is how good the college is placement wise and freq enc is how common is this college

def target_encode_with_smoothing(df, col, target, n_splits=5, smoothing=10):
    global_mean = df[target].mean()
    encoded = pd.Series(index=df.index, dtype=float)
    
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    for train_idx, val_idx in kf.split(df):
        train_fold = df.iloc[train_idx]
        
        agg = train_fold.groupby(col)[target].agg(['mean', 'count'])
        
        smoother = 1 / (1 + np.exp(-(agg['count'] - smoothing)))
        agg['smoothed'] = smoother * agg['mean'] + (1 - smoother) * global_mean
        
        encoded.iloc[val_idx] = df.iloc[val_idx][col].map(agg['smoothed'])
    
    encoded.fillna(global_mean, inplace=True)
    return encoded

def frequency_encode(df, col):
    freq = df[col].value_counts() / len(df)
    return df[col].map(freq)

data['college_target_enc'] = target_encode_with_smoothing(
    data,
    col='college_name',
    target='target',
    smoothing=10
)

data['college_freq_enc'] = frequency_encode(data, 'college_name')

# Drop original column (optional)
data.drop(columns=['college_name'], inplace=True)

In [10]:
degree_encoder = LabelEncoder()
degree_encoder.fit(data['stream'])
data['stream_encoded'] = degree_encoder.transform(data['stream'])

In [11]:
data.head()

,id,name,gender,age,degree,stream,placement_status,salary,gpa,years_of_experience,target,stream_encoded,salary>52500.0,college_target_enc,college_freq_enc
0,1,John Doe,Male,25,Bachelor's,Computer Science,Placed,60000,7.4,2.0,1,0,1,0.814286,0.001429
1,2,Jane Smith,Female,24,Bachelor's,Electrical Engineering,Placed,65000,7.2,1.0,1,1,1,0.814286,0.001429
2,3,Michael Johnson,Male,26,Bachelor's,Mechanical Engineering,Placed,58000,7.6,3.0,1,4,1,0.814286,0.001429
3,4,Emily Davis,Female,23,Bachelor's,Information Technology,Not Placed,0,7.0,2.0,0,3,0,0.814286,0.001429
4,5,David Brown,Male,24,Bachelor's,Computer Science,Placed,62000,7.8,2.0,1,0,1,0.814286,0.001429


In [12]:
train_data = data[['age','gpa','years_of_experience','target','stream_encoded','salary>52500.0','college_target_enc']]

In [13]:
x_train,x_test,y_train,y_test = train_test_split(train_data[['age', 'gpa', 'years_of_experience','stream_encoded','salary>52500.0', 'college_target_enc']],train_data['target'],test_size=.2)

In [16]:
model = XGBClassifier()
model.fit(x_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [18]:
y_pred = model.predict(x_test)

In [20]:
from sklearn.metrics import accuracy_score

In [21]:
accuracy_score(y_pred,y_test)

1.0

In [23]:
model.save_model("model.json")